# Notebook 03b — TabSyn **VAE** training (Colab / overnight)

Runs the **VAE stage only** of TabSyn on Colab's GPU, from the ASUS TUF browser.
Diffusion is intentionally **not** started here (see project `DECISIONS.md`).

### What this notebook does
1. Mounts Google Drive (all outputs persist there — survives a session drop).
2. Clones the CAPSTONE repo + TabSyn (pinned base commit) and applies our patches.
3. Loads the already-extracted **real** features (`X_real.npy`, 22 classes) from Drive.
4. **The 45-class fix:** generates balanced kill-chain sessions for all 45 micro-states,
   gives each a realistic `command_text` (so the semantic feature block isn't all-zeros),
   extracts their 128-d features, and merges them with the real features.
5. **Asserts all 45 classes are present** before any training starts.
6. Trains the VAE (`batch_size=4096`, `epochs=500`), checkpointing every 50 epochs and
   logging epoch/loss/time every epoch to `Drive/.../tabsyn_status.txt`.
7. Stops after the VAE. Does **not** auto-start diffusion.

> **Runtime:** set **Runtime → Change runtime type → GPU** before running.

## 0 · Config — edit these two lines if your paths differ

In [ ]:
# GitHub is the single source of truth for code. Pull it fresh every run.
GITHUB_REPO   = 'https://github.com/MKD2004/adaptive-honeypot-ml-CAPSTONE.git'
GITHUB_BRANCH = 'main'

# Where your project data lives on Google Drive (create this folder and upload
# X_real.npy, y_real.npy, semantic_pca.pkl into it before running).
DRIVE_PROJECT = '/content/drive/MyDrive/capstone'

# TabSyn base commit our patch was made against (pinned so the patch always applies).
TABSYN_BASE_COMMIT = 'cb5ac0f74ec36ee88e7a974a393dfbef50d42da7'

# Training config (mission spec).
VAE_BATCH     = 4096
VAE_EPOCHS    = 500
CKPT_EVERY    = 50      # checkpoint interval (epochs) -> 10 checkpoints over 500
SEED          = 42
SIM_N_TOTAL   = 180_000 # balanced kill-chain sessions (4,000 per class x 45)

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.isdir(DRIVE_PROJECT), (
    f'ABORT: {DRIVE_PROJECT} not found on Drive. Create it and upload '
    'X_real.npy, y_real.npy, semantic_pca.pkl there first.')

# Persistent output locations on Drive (survive a session drop).
DRIVE_OUT   = os.path.join(DRIVE_PROJECT, 'tabsyn_vae_run')
CKPT_DIR    = os.path.join(DRIVE_OUT, 'checkpoints')
STATUS_FILE = os.path.join(DRIVE_OUT, 'tabsyn_status.txt')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Drive project :', DRIVE_PROJECT)
print('Outputs ->     ', DRIVE_OUT)
print('Status file -> ', STATUS_FILE)

## 2 · Clone repo + TabSyn, apply patches, install deps

The patched TabSyn code (4GB-VRAM fixes, CLI-arg wiring, the `zero` shim, and the
Drive checkpoint/epoch-logging hook) is **not** stored in the repo's `tabsyn/`
(that dir is git-ignored) — it lives as `patches/tabsyn-colab.patch` and is applied
onto a pinned upstream commit here.

In [ ]:
import subprocess, sys, os

def sh(cmd, cwd=None):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       capture_output=True)
    if r.stdout: print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError(f'cmd failed: {cmd}')
    return r.stdout

os.chdir('/content')
sh(f'rm -rf capstone tabsyn')
sh(f'git clone --depth 1 -b {GITHUB_BRANCH} {GITHUB_REPO} capstone')

REPO   = '/content/capstone'
PATCH  = f'{REPO}/patches/tabsyn-colab.patch'
assert os.path.isfile(PATCH), f'ABORT: {PATCH} missing — did you push it to GitHub?'

# Clone upstream TabSyn, pin to the exact base commit, apply our patch.
sh('git clone https://github.com/amazon-science/tabsyn.git /content/tabsyn')
sh(f'git checkout {TABSYN_BASE_COMMIT}', cwd='/content/tabsyn')
sh(f'git apply --check "{PATCH}"', cwd='/content/tabsyn')  # fails loudly if it won't apply
sh(f'git apply "{PATCH}"', cwd='/content/tabsyn')
assert os.path.isfile('/content/tabsyn/zero.py'), 'zero shim missing after patch'
print('TabSyn patched OK')

In [ ]:
# TabSyn's module-load import chain needs these; the rest of Colab (torch, numpy,
# pandas, sklearn, transformers) is already present. We deliberately do NOT install
# PyPI 'zero' — the patched tabsyn/zero.py shim shadows it.
sh('pip -q install icecream category_encoders tomli tomli-w ml_collections')
print('deps installed')

## 3 · Load already-extracted real features (22 classes) from Drive

We reuse `X_real.npy` (extracted in notebook 02) rather than re-running DistilBERT on
737k real command strings — the columns are identical to what we extract for the
simulated rows below, so they concatenate directly. `semantic_pca.pkl` is required to
project the simulated `command_text` into the **same** 30-d semantic space.

In [ ]:
import numpy as np, shutil

for fn in ['X_real.npy', 'y_real.npy', 'semantic_pca.pkl']:
    src = os.path.join(DRIVE_PROJECT, fn)
    assert os.path.isfile(src), f'ABORT: {fn} not found in {DRIVE_PROJECT}'

# The extractors look for the PCA at data/processed/semantic_pca.pkl (repo-relative).
os.makedirs(f'{REPO}/honeypot_dataset/data/processed', exist_ok=True)
shutil.copy(os.path.join(DRIVE_PROJECT, 'semantic_pca.pkl'),
            f'{REPO}/honeypot_dataset/data/processed/semantic_pca.pkl')

X_real = np.load(os.path.join(DRIVE_PROJECT, 'X_real.npy'))
y_real = np.load(os.path.join(DRIVE_PROJECT, 'y_real.npy'))
print('X_real:', X_real.shape, ' y_real:', y_real.shape)
print('real classes present:', len(np.unique(y_real)), '/45')

## 4 · THE 45-CLASS FIX — simulate, fill command_text, extract, merge

`generate_balanced_sessions` emits balanced skeletons for **all 45** micro-states.
`add_command_text` gives each a realistic MITRE-mapped command line so the semantic
block (features 76–105, 30 of 128) carries real per-class signal instead of being
all-zeros — which would otherwise let a downstream model trivially separate the
synthetic-only classes by an artifact rather than a learned distribution.

In [ ]:
sys.path.insert(0, f'{REPO}/honeypot_dataset')
os.chdir(f'{REPO}/honeypot_dataset')

from src.generators.kill_chain_simulator import generate_balanced_sessions
from src.generators.sim_commands import add_command_text
from src.extractors.semantic import extract_semantic_batch
from src.extractors.pipeline import build_feature_matrix

# 1) balanced skeletons for all 45 states
df_sim = generate_balanced_sessions(n_total=SIM_N_TOTAL, seed=SEED)
print('simulated sessions:', len(df_sim), ' classes:', df_sim['micro_state'].nunique())

# 2) leak fix — realistic command_text per micro-state
df_sim = add_command_text(df_sim, seed=SEED)
nonempty = int((df_sim['command_text'].str.len() > 0).sum())
print('command_text populated:', nonempty, '/', len(df_sim))

# 3) semantic projection (DistilBERT -> PCA30) on Colab GPU, once, batched
sem_sim = extract_semantic_batch(df_sim['command_text'].tolist())
print('semantic matrix:', sem_sim.shape,
      ' nonzero rows:', int((np.abs(sem_sim).sum(1) > 0).sum()))

# 4) full 128-d extraction for the simulated rows
X_sim, y_sim = build_feature_matrix(df_sim, semantic_matrix=sem_sim, cache_kev=True)
print('X_sim:', X_sim.shape, ' sim classes:', len(np.unique(y_sim)))

# 5) merge real + simulated
X_train_full = np.vstack([X_real, X_sim]).astype(np.float32)
y_train_full = np.concatenate([y_real, y_sim]).astype(np.int64)
print('MERGED  X:', X_train_full.shape, ' y:', y_train_full.shape)
print('merged classes present:', len(np.unique(y_train_full)), '/45')

## 5 · Hard gate — abort unless all 45 classes are present

In [ ]:
present = len(np.unique(y_train_full))
assert present == 45, (
    f'ABORT: only {present}/45 classes present after merge. '
    'Do NOT start training — check the simulator ran and merged correctly.')
print(f'OK — all {present}/45 micro-states present. Safe to train.')

## 6 · Prepare TabSyn input (45-class, balanced)

Same on-disk contract as notebook 03's data-prep cell, but built from the merged
45-class matrix. Rare classes are oversampled so TabSyn sees balanced input.

In [ ]:
import json, pandas as pd
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from configs.schema import FEAT_NAMES

DATANAME = 'honeypot_sessions'
TABSYN_DATA_DIR = f'/content/tabsyn/data/{DATANAME}'
os.makedirs(TABSYN_DATA_DIR, exist_ok=True)

df_input = pd.DataFrame(X_train_full, columns=FEAT_NAMES)
df_input['micro_state'] = y_train_full

target_per_class = max(500, len(df_input) // 45)
frames = []
for cls in sorted(df_input['micro_state'].unique()):
    sub = df_input[df_input['micro_state'] == cls]
    if len(sub) < target_per_class:
        sub = resample(sub, replace=True, n_samples=target_per_class, random_state=SEED)
    frames.append(sub)
df_bal = pd.concat(frames).sample(frac=1, random_state=SEED).reset_index(drop=True)
print('balanced rows:', len(df_bal), ' classes:', df_bal['micro_state'].nunique())

X_all = df_bal[FEAT_NAMES].values.astype(np.float32)
y_all = df_bal['micro_state'].values.astype(np.int64)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.1, random_state=SEED, stratify=y_all)

np.save(f'{TABSYN_DATA_DIR}/X_num_train.npy', X_tr)
np.save(f'{TABSYN_DATA_DIR}/X_num_test.npy',  X_te)
np.save(f'{TABSYN_DATA_DIR}/y_train.npy',     y_tr)
np.save(f'{TABSYN_DATA_DIR}/y_test.npy',      y_te)
np.save(f'{TABSYN_DATA_DIR}/X_cat_train.npy', np.empty((len(X_tr), 0), dtype=str))
np.save(f'{TABSYN_DATA_DIR}/X_cat_test.npy',  np.empty((len(X_te), 0), dtype=str))

cols_all = FEAT_NAMES + ['micro_state']
df_train = pd.DataFrame(np.column_stack([X_tr, y_tr]), columns=cols_all)
df_test  = pd.DataFrame(np.column_stack([X_te, y_te]), columns=cols_all)
df_train.to_csv(f'{TABSYN_DATA_DIR}/train.csv', index=False)
df_test.to_csv(f'{TABSYN_DATA_DIR}/test.csv',   index=False)

info = {
    'name': DATANAME, 'task_type': 'multiclass', 'header': 0,
    'column_names': cols_all, 'num_col_idx': list(range(128)),
    'cat_col_idx': [], 'target_col_idx': [128], 'n_classes': 45,
    'train_num': len(X_tr), 'test_num': len(X_te),
}
with open(f'{TABSYN_DATA_DIR}/info.json', 'w') as f:
    json.dump(info, f, indent=2)

synth_out = f'/content/tabsyn/synthetic/{DATANAME}'
os.makedirs(synth_out, exist_ok=True)
df_train.to_csv(f'{synth_out}/real.csv', index=False)
df_test.to_csv(f'{synth_out}/test.csv', index=False)
print('TabSyn data prepared:', TABSYN_DATA_DIR)
print('  X_num_train:', X_tr.shape, ' classes:', len(np.unique(y_tr)), '/45')

## 7 · Train the VAE (500 epochs) — VAE only, no diffusion

Env vars drive the patched `vae/main.py`'s Drive logging:
`TABSYN_EPOCH_LOG` (per-epoch line to Drive), `TABSYN_CKPT_DIR` + `TABSYN_CKPT_EVERY`
(checkpoint every 50 epochs to Drive). We invoke `main.py --method vae` **only** —
diffusion is a separate, explicitly-greenlit step.

In [ ]:
import time, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_mem/1e9, 1),
      'GB' if torch.cuda.is_available() else '')

env = dict(os.environ)
env['TABSYN_EPOCH_LOG'] = STATUS_FILE
env['TABSYN_CKPT_DIR']  = CKPT_DIR
env['TABSYN_CKPT_EVERY'] = str(CKPT_EVERY)

# Record the pre-flight facts into the status file (survives on Drive).
with open(STATUS_FILE, 'w') as f:
    f.write(time.strftime('%Y-%m-%d %H:%M:%S') +
            f' | START VAE | 45/45 classes asserted | batch={VAE_BATCH} '
            f'epochs={VAE_EPOCHS} ckpt_every={CKPT_EVERY} '
            f'gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}\n')

cmd = [sys.executable, 'main.py',
       '--dataname', DATANAME, '--method', 'vae', '--mode', 'train',
       '--epochs', str(VAE_EPOCHS),
       '--training_batch_size', str(VAE_BATCH)]
print('cmd:', ' '.join(cmd))

t0 = time.time()
proc = subprocess.Popen(cmd, cwd='/content/tabsyn', env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:          # stream training output live
    print(line, end='')
proc.wait()
elapsed = (time.time() - t0) / 60

with open(STATUS_FILE, 'a') as f:
    status = 'SUCCESS' if proc.returncode == 0 else f'FAILED(exit={proc.returncode})'
    f.write(time.strftime('%Y-%m-%d %H:%M:%S') +
            f' | VAE {status} | {elapsed:.1f} min total\n')
assert proc.returncode == 0, f'VAE training failed (exit {proc.returncode})'
print(f'\nVAE training finished in {elapsed:.1f} min')

## 8 · Copy final VAE artifacts to Drive (persist everything)

In [ ]:
import shutil, glob
ckpt_src = f'/content/tabsyn/tabsyn/vae/ckpt/{DATANAME}'
for fn in ['model.pt', 'encoder.pt', 'decoder.pt', 'train_z.npy']:
    p = os.path.join(ckpt_src, fn)
    if os.path.isfile(p):
        shutil.copy(p, os.path.join(DRIVE_OUT, fn))
        print('saved ->', os.path.join(DRIVE_OUT, fn))
    else:
        print('WARN missing:', p)

print('\nCheckpoints on Drive:')
for p in sorted(glob.glob(os.path.join(CKPT_DIR, 'model_epoch*.pt'))):
    print('  ', os.path.basename(p))
print('\n--- tabsyn_status.txt (tail) ---')
with open(STATUS_FILE) as f:
    print(''.join(f.readlines()[-15:]))
print('\nDONE. VAE only — diffusion NOT started (by design).')